# 03 — Limpieza y restauración del dataset · ScoutAI / OFFBALL

Este notebook continúa el ensuciado controlado. El objetivo es **eliminar las filas duplicadas añadidas y restaurar los valores modificados** usando `registro_ensuciado.csv`.

Se trabaja sobre `dataset_prueba_sucio` y el resultado se guarda en `dataset_limpio`. **Los CSV originales no se modifican.** Al final se comparará la copia limpia con los originales.

## 1. Preparar las rutas
Conectamos Google Drive y comprobamos que existan las carpetas necesarias.

In [ ]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

OFFBALL_DIR = Path('/content/drive/MyDrive/PROYECTO-IGOR-CARMELO-BYRON/OFFBALL')
RAW_DIR = OFFBALL_DIR / 'csv_originales'
DIR_SUCIO = OFFBALL_DIR / 'dataset_prueba_sucio'
DIR_LIMPIO = OFFBALL_DIR / 'dataset_limpio'
LOG_PATH = DIR_SUCIO / 'registro_ensuciado.csv'
DIR_LIMPIO.mkdir(parents=True, exist_ok=True)


ARCHIVOS_ESPERADOS = [
    'appearances.csv',
    'clubs.csv',
    'games.csv',
    'player_valuations.csv',
    'players.csv'
]


for ruta in [RAW_DIR, DIR_SUCIO, LOG_PATH]:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encuentra: {ruta}')

for nombre in ARCHIVOS_ESPERADOS:
    if not (RAW_DIR / nombre).exists():
        raise FileNotFoundError(f'Falta el CSV original: {RAW_DIR / nombre}')
    if not (DIR_SUCIO / nombre).exists():
        raise FileNotFoundError(f'Falta el CSV ensuciado: {DIR_SUCIO / nombre}')

print('Carpetas comprobadas.')
print('Dataset limpio se guardará en:', DIR_LIMPIO)

## 2. Cargar el registro de cambios
El notebook 02 generó un registro con estas columnas: `tabla`, `problema`, `fila_indice`, `columna`, `valor_original` y `valor_nuevo`. Usamos estos nombres reales para evitar errores al detectar la fila que hay que restaurar.

In [ ]:
registro = pd.read_csv(LOG_PATH, dtype=str, keep_default_na=False)

COLUMNAS_REGISTRO = ['tabla', 'problema', 'fila_indice', 'columna', 'valor_original', 'valor_nuevo']
faltan = [col for col in COLUMNAS_REGISTRO if col not in registro.columns]
if faltan:
    raise ValueError(f'Faltan columnas en registro_ensuciado.csv: {faltan}. Columnas encontradas: {registro.columns.tolist()}')

print('Columnas del registro:', registro.columns.tolist())
print('Cambios registrados:', len(registro))
display(registro.head())
display(registro['problema'].value_counts().rename_axis('problema').reset_index(name='cantidad'))

## 3. Restaurar los valores y retirar los duplicados añadidos

Se restauran los cambios de cada fila usando `fila_indice` y `valor_original`. En `appearances.csv`, los duplicados se añadieron al final del archivo: se comprueba que su cantidad y contenido correspondan al registro y después se retira únicamente ese bloque añadido, sin eliminar posibles duplicados que ya estuvieran en los datos originales.

In [ ]:
def normalizar_nombre_archivo(valor):
    return Path(str(valor).strip()).name.lower()


def limpiar_csv(nombre_archivo):
    original_path = RAW_DIR / nombre_archivo
    sucio_path = DIR_SUCIO / nombre_archivo
    limpio_path = DIR_LIMPIO / nombre_archivo

    # Se leen como texto para no cambiar formatos ni convertir valores durante la limpieza.
    original = pd.read_csv(original_path, dtype=str, keep_default_na=False)
    sucio = pd.read_csv(sucio_path, dtype=str, keep_default_na=False)


    eventos = registro[registro['tabla'].apply(normalizar_nombre_archivo) == nombre_archivo.lower()].copy()
    eventos_duplicados = eventos[eventos['problema'].str.lower() == 'duplicado'].copy()
    eventos_valor = eventos[eventos['problema'].str.lower() != 'duplicado'].copy()


    # Comprobar filas antes de aplicar cambios: las tablas sin duplicados deben conservar su tamaño.
    filas_originales = len(original)
    cantidad_duplicados_log = len(eventos_duplicados)
    filas_extra = len(sucio) - filas_originales


    if nombre_archivo.lower() == 'appearances.csv':
        if filas_extra != cantidad_duplicados_log:
            raise ValueError(
                f'appearances.csv: se esperaban {cantidad_duplicados_log} filas añadidas según el log, '
                f'pero se encontraron {filas_extra}. No se realizará la limpieza para evitar pérdida de datos.'
            )
        if cantidad_duplicados_log:
            indices_duplicados = pd.to_numeric(eventos_duplicados['fila_indice'], errors='coerce')
            if indices_duplicados.isna().any():
                raise ValueError('El registro contiene índices inválidos para los duplicados de appearances.csv.')
            indices_duplicados = indices_duplicados.astype(int).tolist()
            if any(i < 0 or i >= filas_originales for i in indices_duplicados):
                raise IndexError('Un índice de duplicado no corresponde a una fila original de appearances.csv.')
            bloque_esperado = original.iloc[indices_duplicados].reset_index(drop=True)
            bloque_anadido = sucio.iloc[filas_originales:].reset_index(drop=True)
            if not bloque_esperado.equals(bloque_anadido):
                raise ValueError(
                    'Las filas añadidas de appearances.csv no coinciden con las copias esperadas según el registro. '
                    'Se detiene la limpieza para revisar el archivo.'
                )
        elif filas_extra != 0:
            raise ValueError('appearances.csv tiene filas extra, pero el registro no indica duplicados.')

        # Los duplicados fueron añadidos al final; se conserva el bloque original.
        sucio = sucio.iloc[:filas_originales].copy().reset_index(drop=True)
        duplicados_eliminados = filas_extra
    else:
        if len(sucio) != filas_originales:
            raise ValueError(
                f'{nombre_archivo}: el número de filas difiere del original y el log no registra duplicados. '
                'Se detiene para evitar cambios no documentados.'
            )
        duplicados_eliminados = 0


    valores_restaurados = 0
    for _, evento in eventos_valor.iterrows():
        columna = evento['columna']
        try:
            fila = int(evento['fila_indice'])
        except (ValueError, TypeError):
            raise ValueError(f'Índice de fila inválido en el registro: {evento["fila_indice"]}')
        if columna not in sucio.columns:
            raise KeyError(f'La columna {columna} no existe en {nombre_archivo}.')
        if fila < 0 or fila >= filas_originales:
            raise IndexError(f'El índice {fila} no corresponde a una fila original de {nombre_archivo}.')
        sucio.at[fila, columna] = evento['valor_original']
        valores_restaurados += 1

    sucio.to_csv(limpio_path, index=False)
    return {
        'archivo': nombre_archivo,
        'filas_originales': filas_originales,
        'filas_ensuciadas': len(pd.read_csv(sucio_path, dtype=str, keep_default_na=False)),
        'filas_limpias': len(sucio),
        'duplicados_eliminados': duplicados_eliminados,
        'valores_restaurados': valores_restaurados,
        'archivo_salida': str(limpio_path)
    }


resumen_limpieza = []
for nombre in ARCHIVOS_ESPERADOS:
    resultado = limpiar_csv(nombre)
    resumen_limpieza.append(resultado)
    print(f'Procesado: {nombre}')

resumen_limpieza_df = pd.DataFrame(resumen_limpieza)
display(resumen_limpieza_df)

## 4. Validar la limpieza comparando con los originales

Se comprueba cada archivo limpio contra el original: número de filas, columnas y contenido. Si hay diferencias, se muestran algunas para poder revisarlas.

In [ ]:
verificaciones = []
diferencias_muestra = []

for nombre in ARCHIVOS_ESPERADOS:
    original = pd.read_csv(RAW_DIR / nombre, dtype=str, keep_default_na=False)
    limpio = pd.read_csv(DIR_LIMPIO / nombre, dtype=str, keep_default_na=False)

    mismas_columnas = list(original.columns) == list(limpio.columns)
    mismas_filas = len(original) == len(limpio)
    diferencias_por_columna = {}
    ejemplos = []

    if mismas_columnas and mismas_filas:
        for columna in original.columns:
            distintos = original[columna] != limpio[columna]
            cantidad = int(distintos.sum())
            if cantidad:
                diferencias_por_columna[columna] = cantidad
                indices = distintos[distintos].index[:5]
                for indice in indices:
                    ejemplos.append({
                        'archivo': nombre,
                        'fila': int(indice),
                        'columna': columna,
                        'valor_original': original.at[indice, columna],
                        'valor_limpio': limpio.at[indice, columna]
                    })
    elif mismas_columnas and not mismas_filas:
        diferencias_por_columna['filas'] = abs(len(original) - len(limpio))

    coincidencia_exacta = mismas_columnas and mismas_filas and not diferencias_por_columna
    verificaciones.append({
        'archivo': nombre,
        'filas_originales': len(original),
        'filas_limpias': len(limpio),
        'columnas_coinciden': mismas_columnas,
        'filas_coinciden': mismas_filas,
        'coincidencia_exacta': coincidencia_exacta,
        'diferencias_por_columna': str(diferencias_por_columna)
    })
    diferencias_muestra.extend(ejemplos)

verificacion_df = pd.DataFrame(verificaciones)
display(verificacion_df)

VERIFICACION_PATH = DIR_LIMPIO / 'verificacion_limpieza.csv'
verificacion_df.to_csv(VERIFICACION_PATH, index=False)
print('Verificación guardada en:', VERIFICACION_PATH)

if diferencias_muestra:
    print('Ejemplos de diferencias (máximo 5 por columna):')
    display(pd.DataFrame(diferencias_muestra))

## 5. Resultado

La limpieza se considera correcta solo si la columna `coincidencia_exacta` es `True` para los cinco CSV. El archivo de verificación se guarda en `dataset_limpio/verificacion_limpieza.csv`. Si algún archivo no coincide, revisa las diferencias mostradas antes de continuar.


Una vez confirmada la restauración, el siguiente paso del proyecto será el análisis exploratorio (EDA) y la preparación de las variables para Machine Learning. Los problemas que ya existían en los originales se mantienen sin corregir de forma arbitraria.

In [ ]:
if verificacion_df['coincidencia_exacta'].all():
    print('VALIDACIÓN OK: los cinco CSV limpios coinciden exactamente con los originales.')
else:
    print('VALIDACIÓN PENDIENTE: uno o más CSV todavía presentan diferencias.')
    display(verificacion_df[verificacion_df['coincidencia_exacta'] == False])